<a href="https://colab.research.google.com/github/Shashank-ravi46/Multi-asset-risk-parity-engine-/blob/main/Multi_Asset_Risk_Parity_Engine_%7C_Python%2C_cvxpy%2C_riskfolio_lib%2C_Plotly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q yfinance fredapi riskfolio-lib cvxpy plotly statsmodels scikit-learn kaleido

print("✅ All libraries installed. Ready to build.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.7/451.7 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 93.6 MB/s eta 0:00:00
✅ All libraries installed. Ready to build.


In [6]:
# ============================================================
# CELL 2: SETTINGS AND CONFIGURATION
# ============================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.optimize as sco
from scipy.linalg import inv
import cvxpy as cp

import yfinance as yf
from fredapi import Fred

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

from sklearn.covariance import LedoitWolf
import os, pickle
from datetime import datetime
from IPython.display import display, HTML

# ── Your FRED API key ────────────────────────────────────────
FRED_API_KEY = "098379be7f580ee7b313c29d37080cfb"   # ← Replace this

# ── Date range ───────────────────────────────────────────────
START_DATE      = "2006-01-01"
END_DATE        = datetime.today().strftime("%Y-%m-%d")
ROLLING_WINDOW  = 252       # 1 year of trading days
ANNUALIZATION   = 252
RF_RATE         = 0.05
TARGET_VOL      = 0.10

# ── The 12 assets we will use ────────────────────────────────
ASSET_UNIVERSE = {
    'US_Equity'      : ('SPY',  'US Equity'),
    'Intl_Equity'    : ('EFA',  'Intl Equity'),
    'EM_Equity'      : ('EEM',  'EM Equity'),
    'US_Rates_Long'  : ('TLT',  'US Rates Long'),
    'US_Rates_Short' : ('SHY',  'US Rates Short'),
    'TIPS'           : ('TIP',  'TIPS Inflation'),
    'IG_Credit'      : ('LQD',  'IG Credit'),
    'HY_Credit'      : ('HYG',  'HY Credit'),
    'Commodities'    : ('DJP',  'Commodities'),
    'Oil'            : ('USO',  'Oil'),
    'Gold'           : ('GLD',  'Gold'),
    'REITs'          : ('VNQ',  'REITs'),
}

TICKERS = [v[0] for v in ASSET_UNIVERSE.values()]
LABELS  = {v[0]: v[1] for v in ASSET_UNIVERSE.values()}

ASSET_CLASS_MAP = {
    'SPY': 'equity', 'EFA': 'equity', 'EEM': 'equity',
    'TLT': 'rates',  'SHY': 'rates',  'TIP': 'rates',
    'LQD': 'credit', 'HYG': 'credit',
    'DJP': 'commodity', 'USO': 'commodity',
    'GLD': 'gold',   'VNQ': 'equity'
}

STRESS_PERIODS = {
    'GFC 2008'       : ('2008-09-01', '2009-03-31'),
    'COVID 2020'     : ('2020-02-01', '2020-04-30'),
    'Rate Shock 2022': ('2022-01-01', '2022-12-31'),
}

COLORS = {
    'equity': '#4FC3F7', 'rates': '#81C784',
    'credit': '#FFB74D', 'commodity': '#F06292',
    'gold':   '#FFD54F', 'rp': '#00E5FF',
    'ew':     '#FF6B6B', 'mv': '#A5D6A7',
}

pio.templates.default = "plotly_dark"

print("✅ Settings loaded.")
print(f"   Assets: {TICKERS}")
print(f"   Date range: {START_DATE} → {END_DATE}")
print(f"   FRED key set: {'YES ✅' if FRED_API_KEY != 'YOUR_FRED_API_KEY_HERE' else 'NO ❌ — go back and add your key'}")


✅ Settings loaded.
   Assets: ['SPY', 'EFA', 'EEM', 'TLT', 'SHY', 'TIP', 'LQD', 'HYG', 'DJP', 'USO', 'GLD', 'VNQ']
   Date range: 2006-01-01 → 2026-06-19
   FRED key set: YES ✅


In [7]:
# ============================================================
# CELL 3: CONNECT GOOGLE DRIVE AND CREATE PROJECT FOLDERS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# Create all the folders we need
BASE_DIR = '/content/drive/MyDrive/risk_parity_engine'

DIRS = [
    f'{BASE_DIR}/data/raw',
    f'{BASE_DIR}/data/processed',
    f'{BASE_DIR}/outputs/charts',
]

for d in DIRS:
    os.makedirs(d, exist_ok=True)

print("✅ Google Drive connected and folders created:")
for d in DIRS:
    print(f"   📁 {d}")


Mounted at /content/drive
✅ Google Drive connected and folders created:
   📁 /content/drive/MyDrive/risk_parity_engine/data/raw
   📁 /content/drive/MyDrive/risk_parity_engine/data/processed
   📁 /content/drive/MyDrive/risk_parity_engine/outputs/charts


In [9]:
# ============================================================
# CELL 4: DOWNLOAD ALL DATA
# Yahoo Finance (ETF prices) + FRED (macro data)
# ============================================================

# ── Part 1: Download ETF prices from Yahoo Finance ──────────
print("📥 Downloading ETF prices from Yahoo Finance...")
print("   (This takes 1-2 minutes — please wait)")

try:
    raw = yf.download(
        TICKERS,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=True,
        progress=True
    )['Close']

    # Fill in any missing values
    raw = raw.ffill().bfill()
    raw.dropna(how='all', inplace=True)

    print(f"\n✅ ETF prices downloaded!")
    print(f"   📅 Date range: {raw.index[0].date()} → {raw.index[-1].date()}")
    print(f"   📊 Shape: {raw.shape[0]} days × {raw.shape[1]} assets")
    print(f"   Assets found: {list(raw.columns)}")

except Exception as e:
    print(f"❌ Yahoo Finance failed: {e}")

# ── Part 2: Download macro data from FRED ───────────────────
print("\n📥 Downloading macro data from FRED...")

macro = pd.DataFrame()

if FRED_API_KEY != "YOUR_FRED_API_KEY_HERE":
    try:
        fred = Fred(api_key=FRED_API_KEY)

        fred_series = {
            'UST_10Y'  : 'DGS10',
            'HY_Spread': 'BAMLH0A0HYM2',
            'CPI'      : 'CPIAUCSL',
            'VIX'      : 'VIXCLS',
        }

        frames = {}
        for name, code in fred_series.items():
            try:
                s = fred.get_series(code, observation_start=START_DATE)
                frames[name] = s
                print(f"   ✅ {name} ({code}): {len(s)} data points")
            except Exception as e:
                print(f"   ⚠️  Skipped {name}: {e}")

        macro = pd.DataFrame(frames)
        macro.index = pd.to_datetime(macro.index)
        macro = macro.ffill().resample('D').ffill()
        print(f"\n✅ Macro data downloaded: {macro.shape}")

    except Exception as e:
        print(f"⚠️  FRED error: {e} — continuing without macro data")
else:
    print("⚠️  No FRED key — skipping macro data (project still works fine)")

# ── Save raw data ────────────────────────────────────────────
raw.to_csv(f'{BASE_DIR}/data/raw/etf_prices.csv')
if not macro.empty:
    macro.to_csv(f'{BASE_DIR}/data/raw/fred_macro.csv')

print(f"\n💾 Raw data saved to Google Drive.")



[                       0%                       ]

📥 Downloading ETF prices from Yahoo Finance...
   (This takes 1-2 minutes — please wait)


[*********************100%***********************]  12 of 12 completed



✅ ETF prices downloaded!
   📅 Date range: 2006-01-03 → 2026-06-18
   📊 Shape: 5147 days × 12 assets
   Assets found: ['DJP', 'EEM', 'EFA', 'GLD', 'HYG', 'LQD', 'SHY', 'SPY', 'TIP', 'TLT', 'USO', 'VNQ']

📥 Downloading macro data from FRED...
   ✅ UST_10Y (DGS10): 5338 data points
   ✅ HY_Spread (BAMLH0A0HYM2): 795 data points
   ✅ CPI (CPIAUCSL): 245 data points
   ✅ VIX (VIXCLS): 5338 data points

✅ Macro data downloaded: (7473, 4)

💾 Raw data saved to Google Drive.


In [10]:
# ============================================================
# CELL 5: CLEAN DATA AND CALCULATE RETURNS
# ============================================================

# ── Step 1: Remove assets with not enough history ───────────
print("🧹 Cleaning data...")

min_obs = int(3.0 * 252)  # Need at least 3 years of data per asset

valid = raw.dropna(thresh=min_obs, axis=1)

dropped = set(raw.columns) - set(valid.columns)
if dropped:
    print(f"   ⚠️  Dropped (not enough history): {dropped}")
else:
    print(f"   ✅ All assets have enough history — none dropped")

# Align all assets to same dates (drop days where ANY asset is missing)
valid = valid.dropna(how='any')
print(f"   ✅ Aligned data: {valid.shape[0]} days × {valid.shape[1]} assets")

# ── Step 2: Calculate log returns ───────────────────────────
# Log return = ln(price today / price yesterday)
# We use log returns because they add up correctly over time
returns = np.log(valid / valid.shift(1)).dropna()

ACTIVE_TICKERS = returns.columns.tolist()

print(f"\n✅ Returns calculated!")
print(f"   Assets in model: {ACTIVE_TICKERS}")
print(f"   Return rows: {len(returns)}")

# ── Step 3: Show summary statistics ─────────────────────────
print("\n📊 Annual return statistics per asset:")

stats = pd.DataFrame({
    'Annual Return %' : (returns.mean() * 252 * 100).round(2),
    'Annual Vol %'    : (returns.std() * np.sqrt(252) * 100).round(2),
    'Sharpe Ratio'    : (returns.mean() / returns.std() * np.sqrt(252)).round(3),
})

display(stats)

# ── Save processed returns ───────────────────────────────────
returns.to_csv(f'{BASE_DIR}/data/processed/returns.csv')
print(f"\n💾 Returns saved to Google Drive.")


🧹 Cleaning data...
   ✅ All assets have enough history — none dropped
   ✅ Aligned data: 5147 days × 12 assets

✅ Returns calculated!
   Assets in model: ['DJP', 'EEM', 'EFA', 'GLD', 'HYG', 'LQD', 'SHY', 'SPY', 'TIP', 'TLT', 'USO', 'VNQ']
   Return rows: 5146

📊 Annual return statistics per asset:


,Annual Return %,Annual Vol %,Sharpe Ratio
Ticker,,,
DJP,-0.35,17.65,-0.020
EEM,6.16,27.92,0.220
EFA,5.52,21.41,0.258
GLD,9.73,18.45,0.527
HYG,4.53,10.54,0.430
LQD,4.02,8.64,0.466
SHY,1.95,1.51,1.294
SPY,10.53,19.30,0.546
TIP,3.30,6.20,0.532



💾 Returns saved to Google Drive.


In [11]:
# ============================================================
# CELL 6: COVARIANCE ESTIMATION ENGINE
# How risky is each asset? How do they move together?
# ============================================================

from sklearn.covariance import LedoitWolf

def estimate_covariance(returns_window: pd.DataFrame) -> np.ndarray:
    """
    Estimates how risky each asset is and how they relate to each other.
    Uses Ledoit-Wolf shrinkage — more accurate than simple covariance
    when you have many assets and limited data.
    """
    R = returns_window.values

    # Ledoit-Wolf shrinkage: smarter than raw sample covariance
    lw = LedoitWolf().fit(R)
    cov = lw.covariance_

    # Annualise (daily → yearly)
    cov_ann = cov * 252

    # Make sure matrix is valid (no negative eigenvalues)
    eigvals, eigvecs = np.linalg.eigh(cov_ann)
    eigvals_fixed = np.maximum(eigvals, 1e-8)
    cov_ann = eigvecs @ np.diag(eigvals_fixed) @ eigvecs.T

    return cov_ann


def compute_rolling_covariance(
    returns: pd.DataFrame,
    window: int = 252,
    step: int = 21
) -> dict:
    """
    Computes a fresh covariance matrix every month (every 21 trading days).
    Each matrix looks back 252 days (1 year) to estimate risk.
    """
    cov_history = {}
    dates = returns.index
    n = len(dates)
    count = 0

    print(f"📐 Computing rolling covariance matrices...")
    print(f"   Window: {window} days | Step: {step} days (monthly rebalance)")

    for i in range(window, n, step):
        window_ret = returns.iloc[i - window: i]
        cov = estimate_covariance(window_ret)
        cov_history[dates[i]] = cov
        count += 1

    print(f"   ✅ {count} covariance matrices computed")
    return cov_history


# ── Run the covariance estimation ────────────────────────────
rolling_cov = compute_rolling_covariance(
    returns[ACTIVE_TICKERS],
    window=252,
    step=21
)

# ── Show what one covariance matrix looks like ───────────────
sample_date = list(rolling_cov.keys())[-1]  # Most recent
sample_cov  = rolling_cov[sample_date]

# Convert to correlation for easier reading
vols = np.sqrt(np.diag(sample_cov))
corr = sample_cov / np.outer(vols, vols)
corr_df = pd.DataFrame(corr, index=ACTIVE_TICKERS, columns=ACTIVE_TICKERS).round(2)

print(f"\n📊 Most recent correlation matrix ({sample_date.date()}):")
display(corr_df)

# ── Save to Drive ─────────────────────────────────────────────
with open(f'{BASE_DIR}/data/processed/cov_matrices.pkl', 'wb') as f:
    pickle.dump(rolling_cov, f)

print(f"\n💾 Covariance matrices saved to Google Drive.")

📐 Computing rolling covariance matrices...
   Window: 252 days | Step: 21 days (monthly rebalance)
   ✅ 234 covariance matrices computed

📊 Most recent correlation matrix (2026-06-18):


,DJP,EEM,EFA,GLD,HYG,LQD,SHY,SPY,TIP,TLT,USO,VNQ
DJP,1.00,-0.06,-0.14,0.41,-0.13,-0.17,-0.06,-0.09,0.02,-0.20,0.71,-0.07
EEM,-0.06,1.00,0.78,0.44,0.48,0.37,0.12,0.73,0.19,0.24,-0.38,0.25
EFA,-0.14,0.78,1.00,0.39,0.52,0.43,0.13,0.72,0.21,0.31,-0.45,0.48
GLD,0.41,0.44,0.39,1.00,0.19,0.17,0.09,0.27,0.16,0.12,-0.07,0.14
HYG,-0.13,0.48,0.52,0.19,1.00,0.42,0.13,0.52,0.24,0.29,-0.28,0.37
LQD,-0.17,0.37,0.43,0.17,0.42,1.00,0.19,0.34,0.43,0.66,-0.32,0.37
SHY,-0.06,0.12,0.13,0.09,0.13,0.19,1.00,0.07,0.17,0.19,-0.12,0.13
SPY,-0.09,0.73,0.72,0.27,0.52,0.34,0.07,1.00,0.15,0.16,-0.31,0.34
TIP,0.02,0.19,0.21,0.16,0.24,0.43,0.17,0.15,1.00,0.49,-0.08,0.26
TLT,-0.20,0.24,0.31,0.12,0.29,0.66,0.19,0.16,0.49,1.00,-0.32,0.32



💾 Covariance matrices saved to Google Drive.


In [13]:
# ============================================================
# CELL 7: PORTFOLIO OPTIMIZATION ENGINE
# ERC (Risk Parity), Equal Weight, Mean-Variance
# ============================================================

def optimize_erc(cov: np.ndarray) -> np.ndarray:
    """
    Solves the Equal Risk Contribution (Risk Parity) problem.

    Goal: Find weights so every asset contributes the SAME amount of risk.
    If you have 12 assets, each should contribute exactly 1/12 = 8.33% of risk.

    Uses SLSQP optimizer with 5 random starting points to find global optimum.
    """
    n = cov.shape[0]

    def risk_contributions(w):
        """How much risk is each asset contributing right now?"""
        port_vol = np.sqrt(w @ cov @ w)
        if port_vol <= 0:
            return np.zeros(n)
        marginal = cov @ w
        return w * marginal / port_vol

    def objective(w):
        """
        We want all risk contributions to be equal.
        So we minimise the sum of squared differences from the target.
        When this = 0, perfect risk parity is achieved.
        """
        rc = risk_contributions(w)
        target = rc.sum() / n
        return np.sum((rc - target) ** 2)

    # Rules: weights must be positive, and must sum to 1
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds = [(0.001, 1.0)] * n

    # Try 5 different starting points, keep the best result
    best_w, best_val = None, np.inf
    np.random.seed(42)

    for _ in range(5):
        w0 = np.random.dirichlet(np.ones(n))
        result = sco.minimize(
            objective, w0,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints,
            options={'ftol': 1e-12, 'maxiter': 1000, 'disp': False}
        )
        if result.success and result.fun < best_val:
            best_val = result.fun
            best_w = result.x

    if best_w is None:
        # Fallback: inverse volatility (simple approximation of risk parity)
        vols = np.sqrt(np.diag(cov))
        best_w = (1.0 / vols) / np.sum(1.0 / vols)
        print("   ⚠️  Optimizer failed — using inverse vol fallback")

    return np.maximum(best_w, 0)


def optimize_mean_variance(returns_window: pd.DataFrame, cov: np.ndarray) -> np.ndarray:
    """
    Minimum Variance portfolio using cvxpy.
    Finds the portfolio with the lowest possible volatility.
    """
    n = cov.shape[0]
    w = cp.Variable(n, nonneg=True)
    portfolio_var = cp.quad_form(w, cov)
    constraints = [cp.sum(w) == 1]
    prob = cp.Problem(cp.Minimize(portfolio_var), constraints)

    try:
        prob.solve(solver=cp.OSQP, eps_abs=1e-8, eps_rel=1e-8, verbose=False)
        if w.value is not None:
            return np.maximum(w.value, 0)
        else:
            return np.ones(n) / n
    except:
        return np.ones(n) / n


def equal_weight(n: int) -> np.ndarray:
    """Simple 1/N equal weight — every asset gets the same capital."""
    return np.ones(n) / n


def compute_risk_contributions(w: np.ndarray, cov: np.ndarray) -> np.ndarray:
    """
    Euler decomposition: breaks total portfolio risk into
    each asset's individual contribution.
    For perfect ERC: all values should equal 1/N = 8.33%
    """
    port_vol = np.sqrt(w @ cov @ w)
    if port_vol < 1e-12:
        return np.ones(len(w)) / len(w)
    marginal = cov @ w
    rc = w * marginal / port_vol
    return rc / rc.sum()  # As percentages (sum = 100%)


def apply_vol_targeting(w: np.ndarray, cov: np.ndarray, target: float = 0.10) -> np.ndarray:
    """
    Scales weights up or down so portfolio hits a target volatility of 10%.
    This is how institutional funds apply leverage in risk parity strategies.
    Max leverage capped at 2x for safety.
    """
    port_vol = np.sqrt(w @ cov @ w)
    leverage = min(target / port_vol, 2.0) if port_vol > 1e-8 else 1.0
    return w * leverage


# ── Quick test: show weights for most recent period ──────────
print("🧮 Testing optimization on most recent covariance matrix...")

latest_cov = list(rolling_cov.values())[-1]
n = len(ACTIVE_TICKERS)

w_erc = optimize_erc(latest_cov)
w_ew  = equal_weight(n)
w_mv  = optimize_mean_variance(returns[ACTIVE_TICKERS].tail(252), latest_cov)

rc_erc = compute_risk_contributions(w_erc, latest_cov)

print("\n📊 Current portfolio weights comparison:")
comparison = pd.DataFrame({
    'Asset'           : [LABELS.get(t, t) for t in ACTIVE_TICKERS],
    'Risk Parity %'   : (w_erc * 100).round(1),
    'Equal Weight %'  : (w_ew  * 100).round(1),
    'Min Variance %'  : (w_mv  * 100).round(1),
    'Risk Contrib %'  : (rc_erc * 100).round(1),
}).set_index('Asset')

display(comparison)

print("\n✅ Optimization engine working!")
print(f"   Risk Parity weights sum: {w_erc.sum():.4f} (should be 1.0000)")
print(f"   Risk contributions sum:  {rc_erc.sum():.4f} (should be 1.0000)")


🧮 Testing optimization on most recent covariance matrix...

📊 Current portfolio weights comparison:


,Risk Parity %,Equal Weight %,Min Variance %,Risk Contrib %
Asset,,,,
Commodities,4.3,8.3,0.7,8.3
EM Equity,3.1,8.3,0.0,8.3
Intl Equity,4.1,8.3,0.0,8.3
Gold,2.5,8.3,0.0,8.3
HY Credit,13.1,8.3,23.9,8.3
IG Credit,10.5,8.3,4.9,8.3
US Rates Short,25.1,8.3,47.9,8.3
US Equity,5.4,8.3,0.0,8.3
TIPS Inflation,13.9,8.3,20.6,8.3



✅ Optimization engine working!
   Risk Parity weights sum: 1.0000 (should be 1.0000)
   Risk contributions sum:  1.0000 (should be 1.0000)


In [14]:
# ============================================================
# CELL 8: WALK-FORWARD BACKTESTING ENGINE
# Simulates 20 years of monthly rebalancing
# ============================================================

def run_backtest(
    returns: pd.DataFrame,
    rolling_cov: dict,
    strategy: str = 'erc',
    vol_target: float = None
) -> tuple:
    """
    Simulates the strategy month by month from 2007 to today.

    Each month:
      1. Look at the last 12 months of data
      2. Calculate new optimal weights
      3. Apply those weights to the NEXT month
      4. Record the daily returns

    This is called walk-forward — we never look into the future.
    """
    rebal_dates = sorted(rolling_cov.keys())
    n_assets = len(ACTIVE_TICKERS)

    portfolio_returns = []
    weights_history  = []
    rc_history       = []

    total = len(rebal_dates)
    print(f"⚙️  Running backtest: {strategy.upper()}")
    print(f"   {total} rebalance periods | Monthly rebalancing")

    for i, rebal_date in enumerate(rebal_dates):

        # Show progress every 50 periods
        if i % 50 == 0:
            print(f"   Progress: {i}/{total} periods ({rebal_date.date()})")

        # Forward period: from this rebalance to the next
        next_date = rebal_dates[i + 1] if i + 1 < len(rebal_dates) else returns.index[-1]
        mask = (returns.index > rebal_date) & (returns.index <= next_date)
        fwd_returns = returns.loc[mask, ACTIVE_TICKERS]

        if fwd_returns.empty:
            continue

        cov = rolling_cov[rebal_date]

        # Calculate weights for this period
        try:
            if strategy == 'erc':
                w = optimize_erc(cov)
            elif strategy == 'equal_weight':
                w = equal_weight(n_assets)
            elif strategy == 'mean_variance':
                window_ret = returns.loc[
                    returns.index <= rebal_date, ACTIVE_TICKERS
                ].tail(252)
                w = optimize_mean_variance(window_ret, cov)
            else:
                raise ValueError(f"Unknown strategy: {strategy}")

            # Apply leverage scaling if vol target requested
            if vol_target is not None:
                w = apply_vol_targeting(w, cov, vol_target)

        except Exception as e:
            print(f"   ⚠️  {rebal_date.date()} failed: {e} → equal weight")
            w = equal_weight(n_assets)

        # Portfolio daily return = sum of (weight × asset return)
        port_ret = fwd_returns.values @ w
        rc = compute_risk_contributions(w, cov)

        for date, ret in zip(fwd_returns.index, port_ret):
            portfolio_returns.append({'date': date, 'return': ret})

        weights_history.append(
            dict(zip(['date'] + ACTIVE_TICKERS, [rebal_date] + list(w)))
        )
        rc_history.append(
            dict(zip(['date'] + ACTIVE_TICKERS, [rebal_date] + list(rc)))
        )

    # Build final DataFrames
    ret_series = pd.DataFrame(portfolio_returns).set_index('date')['return']
    weights_df = pd.DataFrame(weights_history).set_index('date')
    rc_df      = pd.DataFrame(rc_history).set_index('date')

    final_nav = np.exp(ret_series.sum())
    print(f"\n   ✅ Done! Final NAV: {final_nav:.3f}x (£100 → £{final_nav*100:.0f})")

    return ret_series, weights_df, rc_df


# ── Run all four strategies ──────────────────────────────────
print("=" * 55)
ret_erc,    wt_erc,    rc_erc    = run_backtest(returns, rolling_cov, 'erc')
print("=" * 55)
ret_ew,     wt_ew,     rc_ew     = run_backtest(returns, rolling_cov, 'equal_weight')
print("=" * 55)
ret_mv,     wt_mv,     rc_mv     = run_backtest(returns, rolling_cov, 'mean_variance')
print("=" * 55)
ret_erc_vt, wt_erc_vt, rc_erc_vt = run_backtest(returns, rolling_cov, 'erc', vol_target=0.10)
print("=" * 55)

# ── Package all strategies ───────────────────────────────────
strategy_returns = {
    'Risk Parity (ERC)'    : ret_erc,
    'Risk Parity (Vol-Tgt)': ret_erc_vt,
    'Equal Weight'         : ret_ew,
    'Min Variance'         : ret_mv,
}

# ── Save weights ─────────────────────────────────────────────
wt_erc.to_csv(f'{BASE_DIR}/outputs/weights_history_erc.csv')
wt_ew.to_csv(f'{BASE_DIR}/outputs/weights_history_ew.csv')
wt_mv.to_csv(f'{BASE_DIR}/outputs/weights_history_mv.csv')

print("\n💾 Weights saved to Google Drive.")

⚙️  Running backtest: ERC
   234 rebalance periods | Monthly rebalancing
   Progress: 0/234 periods (2007-01-05)
   Progress: 50/234 periods (2011-03-08)
   Progress: 100/234 periods (2015-05-11)
   Progress: 150/234 periods (2019-07-12)
   Progress: 200/234 periods (2023-09-13)

   ✅ Done! Final NAV: 1.779x (£100 → £178)
⚙️  Running backtest: EQUAL_WEIGHT
   234 rebalance periods | Monthly rebalancing
   Progress: 0/234 periods (2007-01-05)
   Progress: 50/234 periods (2011-03-08)
   Progress: 100/234 periods (2015-05-11)
   Progress: 150/234 periods (2019-07-12)
   Progress: 200/234 periods (2023-09-13)

   ✅ Done! Final NAV: 2.119x (£100 → £212)
⚙️  Running backtest: MEAN_VARIANCE
   234 rebalance periods | Monthly rebalancing
   Progress: 0/234 periods (2007-01-05)
   Progress: 50/234 periods (2011-03-08)
   Progress: 100/234 periods (2015-05-11)
   Progress: 150/234 periods (2019-07-12)
   Progress: 200/234 periods (2023-09-13)

   ✅ Done! Final NAV: 1.492x (£100 → £149)
⚙️  Runni

In [15]:
# ============================================================
# CELL 9: PERFORMANCE METRICS
# Calculates all the numbers a fund manager would care about
# ============================================================

def compute_performance_metrics(returns_series, rf=0.05, label='Strategy'):
    """
    Computes institutional-grade performance statistics.
    Every number here is something you would see in a real fund factsheet.
    """
    r = returns_series.dropna()
    rf_daily = rf / 252

    # ── Return metrics ───────────────────────────────────────
    cagr   = np.exp(r.mean() * 252) - 1          # Compound annual growth rate
    vol    = r.std() * np.sqrt(252)               # Annualised volatility
    sharpe = (r.mean() - rf_daily) / r.std() * np.sqrt(252)  # Risk-adjusted return

    # ── Drawdown ─────────────────────────────────────────────
    cum_ret     = np.exp(r.cumsum())
    running_max = cum_ret.cummax()
    drawdown    = (cum_ret - running_max) / running_max
    max_dd      = drawdown.min()
    calmar      = cagr / abs(max_dd) if max_dd != 0 else np.nan

    # ── Downside metrics ─────────────────────────────────────
    downside = r[r < rf_daily]
    sortino_denom = downside.std() * np.sqrt(252) if len(downside) > 0 else np.nan
    sortino = (r.mean() - rf_daily) * 252 / sortino_denom if sortino_denom else np.nan

    # ── Tail risk ────────────────────────────────────────────
    var_95  = float(np.percentile(r, 5))           # Worst 5% of days
    cvar_95 = float(r[r <= var_95].mean())          # Average of worst days

    return {
        'Strategy'          : label,
        'CAGR (%)'          : round(cagr * 100, 2),
        'Ann. Vol (%)'      : round(vol * 100, 2),
        'Sharpe Ratio'      : round(sharpe, 3),
        'Max Drawdown (%)'  : round(max_dd * 100, 2),
        'Calmar Ratio'      : round(calmar, 3),
        'Sortino Ratio'     : round(sortino, 3),
        'Skewness'          : round(float(r.skew()), 3),
        'Excess Kurtosis'   : round(float(r.kurtosis()), 3),
        'Daily VaR 95% (%)' : round(var_95 * 100, 3),
        'CVaR / ES 95% (%)' : round(cvar_95 * 100, 3),
        'Hit Rate (%)'      : round((r > 0).mean() * 100, 2),
        'Total Return (%)'  : round((np.exp(r.sum()) - 1) * 100, 2),
    }


# ── Run for all strategies ───────────────────────────────────
metrics_all = pd.DataFrame([
    compute_performance_metrics(ret_erc,    label='Risk Parity (ERC)'),
    compute_performance_metrics(ret_erc_vt, label='Risk Parity (Vol-Tgt)'),
    compute_performance_metrics(ret_ew,     label='Equal Weight'),
    compute_performance_metrics(ret_mv,     label='Min Variance'),
]).set_index('Strategy')

print("📊 FULL PERFORMANCE SUMMARY")
print("=" * 70)
display(metrics_all.T)

metrics_all.to_csv(f'{BASE_DIR}/outputs/performance_summary.csv')
print("\n💾 Performance summary saved to Google Drive.")

📊 FULL PERFORMANCE SUMMARY


Strategy,Risk Parity (ERC),Risk Parity (Vol-Tgt),Equal Weight,Min Variance
CAGR (%),3.010,5.010,3.940,2.080
Ann. Vol (%),5.750,9.940,11.100,2.930
Sharpe Ratio,-0.354,-0.011,-0.102,-1.001
Max Drawdown (%),-19.330,-32.000,-39.280,-11.490
Calmar Ratio,0.156,0.157,0.100,0.181
Sortino Ratio,-0.433,-0.013,-0.123,-1.205
Skewness,-0.853,-1.252,-0.743,-0.667
Excess Kurtosis,15.777,18.991,12.437,29.522
Daily VaR 95% (%),-0.525,-0.888,-0.999,-0.230
CVaR / ES 95% (%),-0.865,-1.491,-1.718,-0.431



💾 Performance summary saved to Google Drive.


In [16]:
# ============================================================
# CELL 10: STRESS TESTING
# How did each strategy survive 2008, 2020, and 2022?
# ============================================================

print("🧪 STRESS TEST ANALYSIS")
print("=" * 70)

stress_results = {}

for period_name, (start, end) in STRESS_PERIODS.items():
    print(f"\n📍 {period_name}  ({start} → {end})")
    print("-" * 50)

    period_data = []
    for name, ret in strategy_returns.items():
        window = ret.loc[start:end]
        if len(window) < 5:
            continue

        # NAV curve for this window
        nav = np.exp(window.cumsum()) * 100
        peak_dd = ((nav / nav.cummax()) - 1).min() * 100

        period_data.append({
            'Strategy'         : name,
            'Total Return (%)'  : round((np.exp(window.sum()) - 1) * 100, 2),
            'Ann. Vol (%)'      : round(window.std() * np.sqrt(252) * 100, 2),
            'Max Drawdown (%)'  : round(peak_dd, 2),
            'Sharpe'            : round(
                window.mean() / window.std() * np.sqrt(252), 3
            ) if window.std() > 0 else np.nan,
        })

    df = pd.DataFrame(period_data).set_index('Strategy')
    display(df)
    stress_results[period_name] = df


🧪 STRESS TEST ANALYSIS

📍 GFC 2008  (2008-09-01 → 2009-03-31)
--------------------------------------------------


,Total Return (%),Ann. Vol (%),Max Drawdown (%),Sharpe
Strategy,,,,
Risk Parity (ERC),-12.66,14.46,-16.37,-1.616
Risk Parity (Vol-Tgt),-21.49,22.95,-26.91,-1.820
Equal Weight,-28.74,32.87,-33.38,-1.779
Min Variance,-5.50,8.99,-10.39,-1.085



📍 COVID 2020  (2020-02-01 → 2020-04-30)
--------------------------------------------------


,Total Return (%),Ann. Vol (%),Max Drawdown (%),Sharpe
Strategy,,,,
Risk Parity (ERC),-6.02,19.42,-14.99,-1.299
Risk Parity (Vol-Tgt),-11.29,34.67,-26.36,-1.404
Equal Weight,-16.83,31.87,-23.44,-2.350
Min Variance,0.78,11.61,-8.15,0.272



📍 Rate Shock 2022  (2022-01-01 → 2022-12-31)
--------------------------------------------------


,Total Return (%),Ann. Vol (%),Max Drawdown (%),Sharpe
Strategy,,,,
Risk Parity (ERC),-10.90,8.77,-14.57,-1.320
Risk Parity (Vol-Tgt),-19.47,14.22,-23.54,-1.529
Equal Weight,-10.58,12.73,-17.04,-0.882
Min Variance,-7.50,4.04,-8.74,-1.938


In [17]:
# ============================================================
# CELL 11: ALL VISUALIZATIONS
# NAV, Risk Contributions, Drawdowns, Frontier, Correlation,
# Ex-Ante vs Realized
# ============================================================

# ── Chart 1: NAV Performance ─────────────────────────────────
print("📈 Building Chart 1: NAV Performance...")

fig1 = go.Figure()
colors_map = {
    'Risk Parity (ERC)'    : '#00E5FF',
    'Risk Parity (Vol-Tgt)': '#00BCD4',
    'Equal Weight'         : '#FF6B6B',
    'Min Variance'         : '#A5D6A7',
}

for name, ret in strategy_returns.items():
    nav = np.exp(ret.cumsum()) * 100
    fig1.add_trace(go.Scatter(
        x=nav.index, y=nav.values, name=name,
        line=dict(color=colors_map.get(name, '#fff'), width=2),
        hovertemplate='%{x|%Y-%m-%d}<br>NAV: %{y:.1f}<extra>' + name + '</extra>'
    ))

for period_name, (s, e) in STRESS_PERIODS.items():
    fig1.add_vrect(
        x0=s, x1=e,
        fillcolor='rgba(255,100,100,0.12)', line_width=0,
        annotation_text=period_name.split()[0],
        annotation_position='top left',
        annotation_font=dict(size=9, color='#ff9999')
    )

fig1.update_layout(
    title='<b>NAV Performance — All Strategies (Base 100)</b>',
    xaxis_title='Date', yaxis_title='NAV (Base 100)',
    height=500, hovermode='x unified',
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(17,17,17,1)',
    legend=dict(x=0.01, y=0.99)
)
fig1.show()
fig1.write_html(f'{BASE_DIR}/outputs/charts/nav_performance.html')
print("   ✅ Chart 1 saved")


# ── Chart 2: Stacked Risk Contributions ──────────────────────
print("📊 Building Chart 2: Risk Contributions...")

fig2 = go.Figure()
asset_colors = ['#4FC3F7','#81C784','#FFB74D','#F06292','#FFD54F',
                '#CE93D8','#80DEEA','#FFCC02','#B0BEC5','#FF8A65',
                '#A5D6A7','#EF9A9A']

for idx, ticker in enumerate(ACTIVE_TICKERS):
    if ticker not in rc_erc.columns:
        continue
    fig2.add_trace(go.Bar(
        x=rc_erc.index,
        y=(rc_erc[ticker] * 100).values,
        name=LABELS.get(ticker, ticker),
        marker_color=asset_colors[idx % len(asset_colors)],
        hovertemplate='%{x|%Y-%m}<br>%{y:.1f}%<extra>'
                      + LABELS.get(ticker, ticker) + '</extra>'
    ))

target_line = 100.0 / len(ACTIVE_TICKERS)
fig2.add_hline(
    y=target_line, line_dash='dot', line_color='white',
    annotation_text=f'Equal target: {target_line:.1f}%',
    annotation_position='right'
)
fig2.update_layout(
    title='<b>ERC Risk Contributions Over Time (%)</b>',
    barmode='stack',
    xaxis_title='Rebalance Date', yaxis_title='Risk Contribution (%)',
    height=450,
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(17,17,17,1)'
)
fig2.show()
fig2.write_html(f'{BASE_DIR}/outputs/charts/risk_contributions.html')
print("   ✅ Chart 2 saved")


# ── Chart 3: Drawdowns ───────────────────────────────────────
print("📉 Building Chart 3: Drawdowns...")

fig3 = go.Figure()
for name, ret in strategy_returns.items():
    cum = np.exp(ret.cumsum())
    dd  = (cum / cum.cummax() - 1) * 100
    fig3.add_trace(go.Scatter(
        x=dd.index, y=dd.values, name=name,
        line=dict(color=colors_map.get(name, '#fff'), width=1.5),
        fill='tozeroy',
        hovertemplate='%{x|%Y-%m-%d}<br>DD: %{y:.1f}%<extra>' + name + '</extra>'
    ))

fig3.update_layout(
    title='<b>Drawdown Analysis — All Strategies</b>',
    xaxis_title='Date', yaxis_title='Drawdown (%)',
    height=400,
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(17,17,17,1)'
)
fig3.show()
fig3.write_html(f'{BASE_DIR}/outputs/charts/drawdowns.html')
print("   ✅ Chart 3 saved")


# ── Chart 4: Efficient Frontier ───────────────────────────────
print("🎯 Building Chart 4: Efficient Frontier...")

r_recent  = returns[ACTIVE_TICKERS].tail(252)
mu_recent = r_recent.mean().values * 252
cov_recent = estimate_covariance(r_recent)
n = len(ACTIVE_TICKERS)

np.random.seed(42)
rnd_vols, rnd_rets, rnd_sharpes = [], [], []
for _ in range(800):
    w = np.random.dirichlet(np.ones(n))
    v = np.sqrt(w @ cov_recent @ w)
    r_val = mu_recent @ w
    rnd_vols.append(v)
    rnd_rets.append(r_val)
    rnd_sharpes.append((r_val - RF_RATE) / v if v > 0 else 0)

fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=rnd_vols, y=rnd_rets, mode='markers',
    marker=dict(color=rnd_sharpes, colorscale='Viridis',
                size=4, opacity=0.5,
                colorbar=dict(title='Sharpe')),
    name='Random Portfolios',
    hovertemplate='Vol: %{x:.2%}<br>Return: %{y:.2%}<extra></extra>'
))

w_erc_now = optimize_erc(cov_recent)
w_ew_now  = equal_weight(n)
w_mv_now  = optimize_mean_variance(r_recent, cov_recent)

for label, w, color, symbol in [
    ('Risk Parity', w_erc_now, '#00E5FF', 'star'),
    ('Equal Weight', w_ew_now, '#FF6B6B', 'circle'),
    ('Min Variance', w_mv_now, '#A5D6A7', 'diamond'),
]:
    v   = np.sqrt(w @ cov_recent @ w)
    ret = mu_recent @ w
    fig4.add_trace(go.Scatter(
        x=[v], y=[ret], mode='markers+text',
        marker=dict(color=color, size=14, symbol=symbol,
                    line=dict(color='white', width=1.5)),
        text=[label], textposition='top right',
        textfont=dict(size=10, color=color),
        name=label
    ))

fig4.update_layout(
    title='<b>Efficient Frontier — Risk vs. Return</b>',
    xaxis_title='Annualised Volatility',
    yaxis_title='Annualised Return',
    xaxis_tickformat='.1%', yaxis_tickformat='.1%',
    height=500,
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(17,17,17,1)'
)
fig4.show()
fig4.write_html(f'{BASE_DIR}/outputs/charts/efficient_frontier.html')
print("   ✅ Chart 4 saved")


# ── Chart 5: Correlation Heatmap ─────────────────────────────
print("🔥 Building Chart 5: Correlation Heatmap...")

corr = returns[ACTIVE_TICKERS].tail(252).corr()
labels_short = [LABELS.get(t, t) for t in corr.columns]

fig5 = go.Figure(go.Heatmap(
    z=corr.values, x=labels_short, y=labels_short,
    colorscale='RdBu_r', zmid=0, zmin=-1, zmax=1,
    text=np.round(corr.values, 2), texttemplate='%{text}',
    colorbar=dict(title='ρ'),
    hovertemplate='%{x} vs %{y}<br>ρ = %{z:.2f}<extra></extra>'
))
fig5.update_layout(
    title='<b>Asset Correlation Matrix (Most Recent 252 Days)</b>',
    height=580, width=720,
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(17,17,17,1)'
)
fig5.show()
fig5.write_html(f'{BASE_DIR}/outputs/charts/correlation_heatmap.html')
print("   ✅ Chart 5 saved")


# ── Chart 6: Ex-Ante vs Realized Risk Contributions ──────────
print("⚖️  Building Chart 6: Ex-Ante vs Realized Risk...")

n = len(ACTIVE_TICKERS)
realized_avg = rc_erc[ACTIVE_TICKERS].mean()
labels_display = [LABELS.get(t, t) for t in ACTIVE_TICKERS]

fig6 = go.Figure()
fig6.add_trace(go.Bar(
    x=labels_display, y=[100.0/n] * n,
    name='Ex-Ante Target', marker_color='#00E5FF',
    hovertemplate='%{x}<br>Target: %{y:.1f}%<extra></extra>'
))
fig6.add_trace(go.Bar(
    x=labels_display, y=(realized_avg * 100).values,
    name='Realized Average', marker_color='#FF6B6B',
    hovertemplate='%{x}<br>Realized: %{y:.1f}%<extra></extra>'
))
fig6.update_layout(
    title='<b>Ex-Ante vs. Realized Risk Contributions (ERC Portfolio)</b>',
    barmode='group',
    xaxis_title='Asset', yaxis_title='Risk Contribution (%)',
    height=450,
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(17,17,17,1)'
)
fig6.show()
fig6.write_html(f'{BASE_DIR}/outputs/charts/exante_vs_realized.html')
print("   ✅ Chart 6 saved")

print("\n✅ Cell 11 complete — all 6 charts built and saved!")


📈 Building Chart 1: NAV Performance...


   ✅ Chart 1 saved
📊 Building Chart 2: Risk Contributions...


   ✅ Chart 2 saved
📉 Building Chart 3: Drawdowns...


   ✅ Chart 3 saved
🎯 Building Chart 4: Efficient Frontier...


   ✅ Chart 4 saved
🔥 Building Chart 5: Correlation Heatmap...


   ✅ Chart 5 saved
⚖️  Building Chart 6: Ex-Ante vs Realized Risk...


   ✅ Chart 6 saved

✅ Cell 11 complete — all 6 charts built and saved!


In [19]:
# ============================================================
# CELL 12: ROLLING AVERAGE CORRELATION OVER TIME
# Shows when diversification broke down (2008, 2020, 2022)
# ============================================================

print("📐 Building rolling correlation chart...")

roll_corr = returns[ACTIVE_TICKERS].rolling(63).corr()
dates_rc  = returns.index[63:]
avg_corr  = []

for date in dates_rc:
    try:
        c = roll_corr.loc[date].values
        n = len(ACTIVE_TICKERS)
        # Extract upper triangle (off-diagonal pairs only)
        idx = np.triu_indices(n, k=1)
        avg_corr.append({
            'date'    : date,
            'avg_corr': c[idx].mean()
        })
    except:
        continue

avg_corr_df = pd.DataFrame(avg_corr).set_index('date')

fig7 = go.Figure()
fig7.add_trace(go.Scatter(
    x=avg_corr_df.index,
    y=avg_corr_df['avg_corr'],
    fill='tozeroy',
    line=dict(color='#00E5FF', width=1.5),
    fillcolor='rgba(0,229,255,0.08)',
    name='Avg Pairwise Correlation',
    hovertemplate='%{x|%Y-%m-%d}<br>Avg Correlation: %{y:.3f}<extra></extra>'
))

for period_name, (s, e) in STRESS_PERIODS.items():
    fig7.add_vrect(
        x0=s, x1=e,
        fillcolor='rgba(255,100,100,0.12)', line_width=0,
        annotation_text=period_name.split()[0],
        annotation_position='top left',
        annotation_font=dict(size=9, color='#ff9999')
    )

fig7.update_layout(
    title='<b>Rolling Average Pairwise Correlation — Diversification Over Time</b>',
    xaxis_title='Date',
    yaxis_title='Average Pairwise Correlation',
    height=420,
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(17,17,17,1)'
)
fig7.show()
fig7.write_html(f'{BASE_DIR}/outputs/charts/rolling_correlation.html')


📐 Building rolling correlation chart...


In [20]:
# ============================================================
# CELL 13: INSTITUTIONAL TEARSHEET — COMBINED DASHBOARD
# One single page showing everything
# ============================================================

print("🎨 Building final tearsheet dashboard...")

fig_ts = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        'NAV Performance (Base 100)',
        'ERC Risk Contributions (%)',
        'Drawdown (%)',
        'Performance Metrics',
        'Asset Correlation Matrix',
        'Ex-Ante vs Realized Risk'
    ],
    specs=[
        [{'type': 'scatter'}, {'type': 'bar'}],
        [{'type': 'scatter'}, {'type': 'table'}],
        [{'type': 'heatmap'}, {'type': 'bar'}],
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

# [1] NAV ─────────────────────────────────────────────────────
for name, ret in strategy_returns.items():
    nav = np.exp(ret.cumsum()) * 100
    fig_ts.add_trace(go.Scatter(
        x=nav.index, y=nav.values, name=name,
        line=dict(color=colors_map.get(name, '#fff'), width=1.5),
        showlegend=True
    ), row=1, col=1)

# [2] Risk Contributions ──────────────────────────────────────
for idx, ticker in enumerate(ACTIVE_TICKERS[:8]):
    if ticker not in rc_erc.columns:
        continue
    fig_ts.add_trace(go.Bar(
        x=rc_erc.index,
        y=(rc_erc[ticker] * 100).values,
        name=LABELS.get(ticker, ticker),
        marker_color=asset_colors[idx % len(asset_colors)],
        showlegend=False
    ), row=1, col=2)

# [3] Drawdowns ───────────────────────────────────────────────
for name, ret in strategy_returns.items():
    cum = np.exp(ret.cumsum())
    dd  = (cum / cum.cummax() - 1) * 100
    fig_ts.add_trace(go.Scatter(
        x=dd.index, y=dd.values, name=name,
        line=dict(color=colors_map.get(name, '#fff'), width=1.2),
        showlegend=False
    ), row=2, col=1)

# [4] Performance Table ───────────────────────────────────────
key_cols = ['CAGR (%)', 'Ann. Vol (%)', 'Sharpe Ratio', 'Max Drawdown (%)']
available_cols = [c for c in key_cols if c in metrics_all.columns]
tbl = metrics_all[available_cols].T.reset_index()

fig_ts.add_trace(go.Table(
    header=dict(
        values=['Metric'] + metrics_all.index.tolist(),
        fill_color='rgba(0,100,150,0.8)',
        font=dict(color='white', size=9),
        align='center'
    ),
    cells=dict(
        values=[tbl.iloc[:, i].tolist() for i in range(len(tbl.columns))],
        fill_color='rgba(30,30,30,0.9)',
        font=dict(color='white', size=8),
        align='center'
    )
), row=2, col=2)

# [5] Correlation Heatmap ─────────────────────────────────────
corr_ts = returns[ACTIVE_TICKERS].tail(252).corr()
labels_ts = [LABELS.get(t, t)[:8] for t in corr_ts.columns]
fig_ts.add_trace(go.Heatmap(
    z=corr_ts.values, x=labels_ts, y=labels_ts,
    colorscale='RdBu_r', zmid=0, showscale=False,
    hovertemplate='%{x} vs %{y}<br>ρ = %{z:.2f}<extra></extra>'
), row=3, col=1)

# [6] Ex-Ante vs Realized ─────────────────────────────────────
realized_avg_ts = rc_erc[ACTIVE_TICKERS].mean()
labels_ts2 = [LABELS.get(t, t)[:8] for t in ACTIVE_TICKERS]
n_ts = len(ACTIVE_TICKERS)

fig_ts.add_trace(go.Bar(
    x=labels_ts2, y=[100.0/n_ts] * n_ts,
    name='Ex-Ante', marker_color='#00E5FF', showlegend=False
), row=3, col=2)
fig_ts.add_trace(go.Bar(
    x=labels_ts2, y=(realized_avg_ts * 100).values,
    name='Realized', marker_color='#FF6B6B', showlegend=False
), row=3, col=2)

fig_ts.update_layout(
    title=dict(
        text='<b>MULTI-ASSET RISK PARITY ENGINE — INSTITUTIONAL TEARSHEET</b>',
        font=dict(size=16), x=0.5, xanchor='center'
    ),
    height=1200, width=1300,
    barmode='group',
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(12,12,20,1)',
    font=dict(color='white'),
    legend=dict(x=0.01, y=0.99, font=dict(size=9))
)

fig_ts.show()
fig_ts.write_html(f'{BASE_DIR}/outputs/charts/tearsheet.html')

try:
    fig_ts.write_image(f'{BASE_DIR}/outputs/charts/tearsheet.png', scale=2)
    print("✅ Tearsheet saved as PNG (for GitHub README)")
except:
    print("⚠️  PNG needs kaleido — HTML version saved successfully")

🎨 Building final tearsheet dashboard...


⚠️  PNG needs kaleido — HTML version saved successfully


In [21]:
# ============================================================
# CELL 14: SAVE ALL FINAL DELIVERABLES
# ============================================================

print("💾 Saving all final deliverables...")

# Strategy returns
pd.DataFrame({
    'ERC'          : ret_erc,
    'ERC_VolTgt'   : ret_erc_vt,
    'EqualWeight'  : ret_ew,
    'MinVariance'  : ret_mv,
}).to_csv(f'{BASE_DIR}/outputs/strategy_returns.csv')

# Weights
wt_erc.to_csv(f'{BASE_DIR}/outputs/weights_history_erc.csv')
wt_ew.to_csv(f'{BASE_DIR}/outputs/weights_history_ew.csv')
wt_mv.to_csv(f'{BASE_DIR}/outputs/weights_history_mv.csv')

# Performance metrics
metrics_all.to_csv(f'{BASE_DIR}/outputs/performance_summary.csv')

# Stress test results
for name, df in stress_results.items():
    safe_name = name.replace(' ', '_').replace('/', '_')
    df.to_csv(f'{BASE_DIR}/outputs/stress_{safe_name}.csv')

print("\n✅ ALL FILES SAVED. Here is everything in your Drive:\n")

for root, dirs, files in os.walk(BASE_DIR):
    level  = root.replace(BASE_DIR, '').count(os.sep)
    indent = '   ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    for f in files:
        sub = '   ' * (level + 1)
        print(f'{sub}📄 {f}')

print("""
╔══════════════════════════════════════════════════════════╗
║        🎉 PROJECT COMPLETE — RISK PARITY ENGINE          ║
╠══════════════════════════════════════════════════════════╣
║  ✅ 12-asset universe: equities, rates, credit,          ║
║     commodities, gold                                    ║
║  ✅ 3 strategies: ERC, Equal Weight, Min Variance        ║
║  ✅ 20-year walk-forward backtest                        ║
║  ✅ Stress tests: 2008, 2020, 2022                       ║
║  ✅ 7 professional Plotly charts                         ║
║  ✅ Institutional tearsheet dashboard                    ║
║  ✅ All outputs saved to Google Drive                    ║
╚══════════════════════════════════════════════════════════╝
""")


💾 Saving all final deliverables...

✅ ALL FILES SAVED. Here is everything in your Drive:

📁 risk_parity_engine/
   📁 data/
      📁 raw/
         📄 etf_prices.csv
         📄 fred_macro.csv
      📁 processed/
         📄 returns.csv
         📄 cov_matrices.pkl
   📁 outputs/
      📄 weights_history_erc.csv
      📄 weights_history_ew.csv
      📄 weights_history_mv.csv
      📄 performance_summary.csv
      📄 strategy_returns.csv
      📄 stress_GFC_2008.csv
      📄 stress_COVID_2020.csv
      📄 stress_Rate_Shock_2022.csv
      📁 charts/
         📄 nav_performance.html
         📄 risk_contributions.html
         📄 drawdowns.html
         📄 efficient_frontier.html
         📄 correlation_heatmap.html
         📄 exante_vs_realized.html
         📄 rolling_correlation.html
         📄 tearsheet.html

╔══════════════════════════════════════════════════════════╗
║        🎉 PROJECT COMPLETE — RISK PARITY ENGINE          ║
╠══════════════════════════════════════════════════════════╣
║  ✅ 12-asset univer

In [22]:
# ============================================================
# CELL 15: EXPORT ALL RESULTS TO EXCEL
# One file, multiple sheets, ready to show recruiters
# ============================================================

import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

output_path = f'{BASE_DIR}/outputs/Risk_Parity_Engine.xlsx'

print("📊 Building Excel file...")

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

    # ── Sheet 1: Performance Summary ─────────────────────────
    metrics_all.to_excel(writer, sheet_name='Performance Summary')
    print("   ✅ Sheet 1: Performance Summary")

    # ── Sheet 2: Strategy Returns ─────────────────────────────
    returns_df = pd.DataFrame({
        'Risk Parity (ERC)'    : ret_erc,
        'Risk Parity (Vol-Tgt)': ret_erc_vt,
        'Equal Weight'         : ret_ew,
        'Min Variance'         : ret_mv,
    })
    returns_df.to_excel(writer, sheet_name='Daily Returns')
    print("   ✅ Sheet 2: Daily Returns")

    # ── Sheet 3: NAV Curves ───────────────────────────────────
    nav_df = pd.DataFrame({
        'Risk Parity (ERC)'    : np.exp(ret_erc.cumsum()) * 100,
        'Risk Parity (Vol-Tgt)': np.exp(ret_erc_vt.cumsum()) * 100,
        'Equal Weight'         : np.exp(ret_ew.cumsum()) * 100,
        'Min Variance'         : np.exp(ret_mv.cumsum()) * 100,
    })
    nav_df.to_excel(writer, sheet_name='NAV Curves')
    print("   ✅ Sheet 3: NAV Curves")

    # ── Sheet 4: ERC Weights History ──────────────────────────
    wt_erc_pct = (wt_erc * 100).round(2)
    wt_erc_pct.columns = [LABELS.get(t, t) for t in wt_erc_pct.columns]
    wt_erc_pct.to_excel(writer, sheet_name='ERC Weights History')
    print("   ✅ Sheet 4: ERC Weights History")

    # ── Sheet 5: Equal Weight History ────────────────────────
    wt_ew_pct = (wt_ew * 100).round(2)
    wt_ew_pct.columns = [LABELS.get(t, t) for t in wt_ew_pct.columns
                         if t in wt_ew_pct.columns]
    wt_ew_pct.to_excel(writer, sheet_name='EW Weights History')
    print("   ✅ Sheet 5: Equal Weight History")

    # ── Sheet 6: Min Variance Weights ────────────────────────
    wt_mv_pct = (wt_mv * 100).round(2)
    wt_mv_pct.columns = [LABELS.get(t, t) for t in wt_mv_pct.columns
                         if t in wt_mv_pct.columns]
    wt_mv_pct.to_excel(writer, sheet_name='MV Weights History')
    print("   ✅ Sheet 6: Min Variance Weights")

    # ── Sheet 7: Risk Contributions ──────────────────────────
    rc_pct = (rc_erc[ACTIVE_TICKERS] * 100).round(2)
    rc_pct.columns = [LABELS.get(t, t) for t in rc_pct.columns]
    rc_pct.to_excel(writer, sheet_name='Risk Contributions')
    print("   ✅ Sheet 7: Risk Contributions")

    # ── Sheet 8: Stress Test Results ─────────────────────────
    stress_combined = []
    for period_name, df in stress_results.items():
        df_copy = df.copy()
        df_copy.insert(0, 'Period', period_name)
        stress_combined.append(df_copy)

    if stress_combined:
        pd.concat(stress_combined).to_excel(
            writer, sheet_name='Stress Tests'
        )
    print("   ✅ Sheet 8: Stress Tests")

    # ── Sheet 9: Current Weights Snapshot ────────────────────
    latest_cov_snap = list(rolling_cov.values())[-1]
    n_snap = len(ACTIVE_TICKERS)

    w_erc_now = optimize_erc(latest_cov_snap)
    w_ew_now  = equal_weight(n_snap)
    w_mv_now  = optimize_mean_variance(
        returns[ACTIVE_TICKERS].tail(252), latest_cov_snap
    )
    rc_now = compute_risk_contributions(w_erc_now, latest_cov_snap)

    snapshot = pd.DataFrame({
        'Asset'              : [LABELS.get(t, t) for t in ACTIVE_TICKERS],
        'Ticker'             : ACTIVE_TICKERS,
        'Risk Parity Wt (%)'  : (w_erc_now * 100).round(2),
        'Equal Weight Wt (%)' : (w_ew_now  * 100).round(2),
        'Min Variance Wt (%)' : (w_mv_now  * 100).round(2),
        'Risk Contrib (%)'    : (rc_now    * 100).round(2),
    }).set_index('Asset')

    snapshot.to_excel(writer, sheet_name='Current Snapshot')
    print("   ✅ Sheet 9: Current Weights Snapshot")

    # ── Sheet 10: Asset Return Statistics ────────────────────
    asset_stats = pd.DataFrame({
        'Annual Return %' : (returns[ACTIVE_TICKERS].mean() * 252 * 100).round(2),
        'Annual Vol %'    : (returns[ACTIVE_TICKERS].std() * np.sqrt(252) * 100).round(2),
        'Sharpe Ratio'    : (returns[ACTIVE_TICKERS].mean() /
                             returns[ACTIVE_TICKERS].std() * np.sqrt(252)).round(3),
        'Max Drawdown %'  : (
            (returns[ACTIVE_TICKERS].cumsum().apply(np.exp) /
             returns[ACTIVE_TICKERS].cumsum().apply(np.exp).cummax() - 1
            ).min() * 100
        ).round(2),
    })
    asset_stats.index = [LABELS.get(t, t) for t in asset_stats.index]
    asset_stats.to_excel(writer, sheet_name='Asset Statistics')
    print("   ✅ Sheet 10: Asset Statistics")

print(f"\n💾 Excel file saved to Google Drive:")
print(f"   {output_path}")
print("\n📥 TO DOWNLOAD:")
print("   1. Look at the LEFT sidebar in Colab")
print("   2. Click the 📁 folder icon")
print("   3. Navigate to: drive → MyDrive → risk_parity_engine → outputs")
print("   4. Right-click 'Risk_Parity_Engine.xlsx'")
print("   5. Click Download")
print("\n✅ Cell 15 complete — Excel file ready!")


📊 Building Excel file...
   ✅ Sheet 1: Performance Summary
   ✅ Sheet 2: Daily Returns
   ✅ Sheet 3: NAV Curves
   ✅ Sheet 4: ERC Weights History
   ✅ Sheet 5: Equal Weight History
   ✅ Sheet 6: Min Variance Weights
   ✅ Sheet 7: Risk Contributions
   ✅ Sheet 8: Stress Tests
   ✅ Sheet 9: Current Weights Snapshot
   ✅ Sheet 10: Asset Statistics

💾 Excel file saved to Google Drive:
   /content/drive/MyDrive/risk_parity_engine/outputs/Risk_Parity_Engine.xlsx

📥 TO DOWNLOAD:
   1. Look at the LEFT sidebar in Colab
   2. Click the 📁 folder icon
   3. Navigate to: drive → MyDrive → risk_parity_engine → outputs
   4. Right-click 'Risk_Parity_Engine.xlsx'
   5. Click Download

✅ Cell 15 complete — Excel file ready!


In [23]:
# ============================================================
# CELL 16A: INSTALL IMAGE EXPORT TOOL
# ============================================================

!pip install -q kaleido

print("✅ kaleido installed — ready to export charts as images")


✅ kaleido installed — ready to export charts as images


In [24]:
# ============================================================
# CELL 16B: EXPORT ALL CHARTS AS PNG + EMBED INTO EXCEL
# ============================================================

from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
import os

print("🖼️  Saving all charts as PNG images...")

# ── Step 1: Save every chart as a PNG file ───────────────────

charts_to_save = {
    'nav_performance'     : fig1,
    'risk_contributions'  : fig2,
    'drawdowns'           : fig3,
    'efficient_frontier'  : fig4,
    'correlation_heatmap' : fig5,
    'exante_vs_realized'  : fig6,
    'rolling_correlation' : fig7,
    'tearsheet'           : fig_ts,
}

png_paths = {}

for name, fig in charts_to_save.items():
    path = f'{BASE_DIR}/outputs/charts/{name}.png'
    try:
        fig.write_image(path, scale=2, width=1200, height=600)
        png_paths[name] = path
        print(f"   ✅ Saved: {name}.png")
    except Exception as e:
        print(f"   ⚠️  Could not save {name}: {e}")

print(f"\n✅ {len(png_paths)} chart images saved")


# ── Step 2: Add each chart as its own sheet in Excel ─────────

print("\n📊 Embedding charts into Excel...")

# Load the existing Excel file we built in Cell 15
wb = load_workbook(output_path)

chart_sheet_names = {
    'nav_performance'     : '📈 NAV Chart',
    'risk_contributions'  : '📊 Risk Contributions',
    'drawdowns'           : '📉 Drawdowns',
    'efficient_frontier'  : '🎯 Efficient Frontier',
    'correlation_heatmap' : '🔥 Correlation Heatmap',
    'exante_vs_realized'  : '⚖️  Ex-Ante vs Realized',
    'rolling_correlation' : '📐 Rolling Correlation',
    'tearsheet'           : '🏆 TEARSHEET',
}

for chart_name, sheet_label in chart_sheet_names.items():
    if chart_name not in png_paths:
        print(f"   ⚠️  Skipping {chart_name} — PNG not found")
        continue

    try:
        # Create a new sheet for this chart
        # Clean sheet name (Excel doesn't allow special chars)
        safe_label = sheet_label.replace('📈','').replace('📊','') \
                                .replace('📉','').replace('🎯','') \
                                .replace('🔥','').replace('⚖️','') \
                                .replace('📐','').replace('🏆','').strip()

        ws = wb.create_sheet(title=safe_label[:31])  # Excel max 31 chars

        # Add a title in cell A1
        ws['A1'] = safe_label
        ws['A1'].font = Font(bold=True, size=14, color='000000')

        # Embed the PNG image starting at cell A3
        img = XLImage(png_paths[chart_name])

        # Scale image to fit nicely in Excel
        img.width  = 900
        img.height = 450
        ws.add_image(img, 'A3')

        # Set row height so image is visible
        ws.row_dimensions[3].height = 340

        print(f"   ✅ Embedded: {safe_label}")

    except Exception as e:
        print(f"   ⚠️  Could not embed {chart_name}: {e}")

# ── Step 3: Move Tearsheet to be the FIRST sheet ─────────────
try:
    tearsheet_sheet = wb['TEARSHEET']
    wb.move_sheet(tearsheet_sheet, offset=-len(wb.sheetnames))
    print("\n   ✅ Tearsheet moved to first position")
except:
    pass

# ── Step 4: Save the updated Excel file ──────────────────────
wb.save(output_path)
wb.close()

print(f"\n💾 Excel file updated with all charts:")
print(f"   {output_path}")

print("""
┌─────────────────────────────────────────────────┐
│           YOUR EXCEL FILE NOW CONTAINS          │
├─────────────────────────────────────────────────┤
│  Tab 1:  TEARSHEET (full dashboard image)       │
│  Tab 2:  Performance Summary (numbers table)    │
│  Tab 3:  Daily Returns                          │
│  Tab 4:  NAV Curves                             │
│  Tab 5:  ERC Weights History                    │
│  Tab 6:  EW Weights History                     │
│  Tab 7:  MV Weights History                     │
│  Tab 8:  Risk Contributions                     │
│  Tab 9:  Stress Tests                           │
│  Tab 10: Current Snapshot                       │
│  Tab 11: Asset Statistics                       │
│  Tab 12: NAV Chart (image)                      │
│  Tab 13: Risk Contributions (image)             │
│  Tab 14: Drawdowns (image)                      │
│  Tab 15: Efficient Frontier (image)             │
│  Tab 16: Correlation Heatmap (image)            │
│  Tab 17: Ex-Ante vs Realized (image)            │
│  Tab 18: Rolling Correlation (image)            │
└─────────────────────────────────────────────────┘
""")

print("📥 TO DOWNLOAD YOUR EXCEL FILE:")
print("   1. Click the 📁 folder icon on the LEFT sidebar")
print("   2. Go to: drive → MyDrive → risk_parity_engine → outputs")
print("   3. Right-click  'Risk_Parity_Engine.xlsx'")
print("   4. Click Download")
print("\n✅ Cell 16 complete — Excel file is fully ready!")


🖼️  Saving all charts as PNG images...
   ⚠️  Could not save nav_performance: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

   ⚠️  Could not save risk_contributions: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

   ⚠️  Could not save drawdowns: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

   ⚠️  Could not save efficient_frontier: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

   ⚠️  Could not save correlation_heatmap: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

   ⚠️  Could not save exante_vs_realized: 
Image export using the "kaleido" engine requ

In [25]:
# ============================================================
# CELL 17: COMPLETE EXCEL FILE — DATA + NATIVE CHARTS
# Uses xlsxwriter to build real Excel charts (not images)
# ============================================================

!pip install -q xlsxwriter
print("✅ xlsxwriter ready")


✅ xlsxwriter ready


In [26]:
# ============================================================
# CELL 17B: BUILD THE FULL EXCEL FILE
# ============================================================

import xlsxwriter
import numpy as np
import pandas as pd

output_excel = f'{BASE_DIR}/outputs/Risk_Parity_Full.xlsx'

print("📊 Building complete Excel file with data + charts...")

wb = xlsxwriter.Workbook(output_excel)

# ── Global formats ────────────────────────────────────────────
fmt_title = wb.add_format({
    'bold': True, 'font_size': 14,
    'font_color': 'white', 'bg_color': '#0A2342',
    'align': 'center', 'valign': 'vcenter',
    'border': 1
})
fmt_header = wb.add_format({
    'bold': True, 'font_size': 10,
    'font_color': 'white', 'bg_color': '#1565C0',
    'align': 'center', 'valign': 'vcenter',
    'border': 1, 'text_wrap': True
})
fmt_subheader = wb.add_format({
    'bold': True, 'font_size': 10,
    'font_color': 'white', 'bg_color': '#0D47A1',
    'align': 'left', 'border': 1
})
fmt_number = wb.add_format({
    'num_format': '0.00', 'border': 1,
    'align': 'center', 'bg_color': '#F5F5F5'
})
fmt_pct = wb.add_format({
    'num_format': '0.00"%"', 'border': 1,
    'align': 'center', 'bg_color': '#F5F5F5'
})
fmt_date = wb.add_format({
    'num_format': 'yyyy-mm-dd', 'border': 1,
    'align': 'center', 'bg_color': '#FAFAFA'
})
fmt_green = wb.add_format({
    'num_format': '0.00', 'border': 1,
    'align': 'center', 'bg_color': '#C8E6C9',
    'font_color': '#1B5E20', 'bold': True
})
fmt_red = wb.add_format({
    'num_format': '0.00', 'border': 1,
    'align': 'center', 'bg_color': '#FFCDD2',
    'font_color': '#B71C1C', 'bold': True
})
fmt_label = wb.add_format({
    'bold': True, 'font_size': 10,
    'bg_color': '#E3F2FD', 'border': 1,
    'align': 'left'
})


# ════════════════════════════════════════════════════════════
# SHEET 1: SUMMARY DASHBOARD
# ════════════════════════════════════════════════════════════
ws1 = wb.add_worksheet('Summary Dashboard')
ws1.set_tab_color('#0A2342')
ws1.set_column('A:A', 28)
ws1.set_column('B:E', 18)
ws1.set_row(0, 35)
ws1.set_row(1, 20)

ws1.merge_range('A1:E1',
    'MULTI-ASSET RISK PARITY ENGINE — PERFORMANCE SUMMARY',
    fmt_title)

headers = ['Metric', 'Risk Parity (ERC)',
           'Risk Parity (Vol-Tgt)', 'Equal Weight', 'Min Variance']
for col, h in enumerate(headers):
    ws1.write(2, col, h, fmt_header)

metrics_rows = [
    'CAGR (%)', 'Ann. Vol (%)', 'Sharpe Ratio',
    'Max Drawdown (%)', 'Calmar Ratio', 'Sortino Ratio',
    'Skewness', 'Excess Kurtosis',
    'Daily VaR 95% (%)', 'CVaR / ES 95% (%)',
    'Hit Rate (%)', 'Total Return (%)'
]

strategies_order = [
    'Risk Parity (ERC)',
    'Risk Parity (Vol-Tgt)',
    'Equal Weight',
    'Min Variance'
]

for row_idx, metric in enumerate(metrics_rows):
    ws1.write(row_idx + 3, 0, metric, fmt_label)
    for col_idx, strat in enumerate(strategies_order):
        if strat in metrics_all.index and metric in metrics_all.columns:
            val = metrics_all.loc[strat, metric]
            # Green for Sharpe/Calmar/Sortino/HitRate, red for drawdown/VaR
            if metric in ['Max Drawdown (%)', 'Daily VaR 95% (%)', 'CVaR / ES 95% (%)']:
                fmt = fmt_red
            elif metric in ['Sharpe Ratio', 'Calmar Ratio', 'Sortino Ratio', 'CAGR (%)']:
                fmt = fmt_green
            else:
                fmt = fmt_number
            ws1.write(row_idx + 3, col_idx + 1, val, fmt)

print("   ✅ Sheet 1: Summary Dashboard")


# ════════════════════════════════════════════════════════════
# SHEET 2: NAV CURVES + CHART
# ════════════════════════════════════════════════════════════
ws2 = wb.add_worksheet('NAV Curves')
ws2.set_tab_color('#1565C0')
ws2.set_column('A:A', 14)
ws2.set_column('B:E', 16)

nav_df = pd.DataFrame({
    'Risk Parity (ERC)'    : np.exp(ret_erc.cumsum()) * 100,
    'Risk Parity (Vol-Tgt)': np.exp(ret_erc_vt.cumsum()) * 100,
    'Equal Weight'         : np.exp(ret_ew.cumsum()) * 100,
    'Min Variance'         : np.exp(ret_mv.cumsum()) * 100,
}).reset_index()
nav_df.columns = ['Date','Risk Parity (ERC)',
                  'Risk Parity (Vol-Tgt)','Equal Weight','Min Variance']

ws2.merge_range('A1:E1', 'NAV PERFORMANCE — BASE 100', fmt_title)
nav_headers = ['Date','Risk Parity (ERC)',
               'Risk Parity (Vol-Tgt)','Equal Weight','Min Variance']
for col, h in enumerate(nav_headers):
    ws2.write(1, col, h, fmt_header)

for row_idx, row in nav_df.iterrows():
    ws2.write_datetime(row_idx + 2, 0,
        pd.Timestamp(row['Date']).to_pydatetime(), fmt_date)
    for col_idx, col_name in enumerate(nav_headers[1:]):
        ws2.write(row_idx + 2, col_idx + 1,
                  round(float(row[col_name]), 4), fmt_number)

# Native Excel line chart
chart2 = wb.add_chart({'type': 'line'})
n_rows = len(nav_df)
colors_xl = ['#00E5FF', '#00BCD4', '#FF6B6B', '#A5D6A7']
names_xl   = ['Risk Parity (ERC)', 'Risk Parity (Vol-Tgt)',
              'Equal Weight', 'Min Variance']

for i, (name, color) in enumerate(zip(names_xl, colors_xl)):
    chart2.add_series({
        'name'      : name,
        'categories': ['NAV Curves', 2, 0, n_rows + 1, 0],
        'values'    : ['NAV Curves', 2, i+1, n_rows + 1, i+1],
        'line'      : {'color': color, 'width': 1.5},
    })

chart2.set_title({'name': 'NAV Performance (Base 100)'})
chart2.set_x_axis({'name': 'Date', 'date_axis': True})
chart2.set_y_axis({'name': 'NAV'})
chart2.set_legend({'position': 'bottom'})
chart2.set_size({'width': 720, 'height': 400})
ws2.insert_chart('G2', chart2)

print("   ✅ Sheet 2: NAV Curves + Chart")


# ════════════════════════════════════════════════════════════
# SHEET 3: DRAWDOWNS + CHART
# ════════════════════════════════════════════════════════════
ws3 = wb.add_worksheet('Drawdowns')
ws3.set_tab_color('#B71C1C')
ws3.set_column('A:A', 14)
ws3.set_column('B:E', 16)

dd_df = pd.DataFrame({
    'Risk Parity (ERC)'    : (np.exp(ret_erc.cumsum()) /
                               np.exp(ret_erc.cumsum()).cummax() - 1) * 100,
    'Risk Parity (Vol-Tgt)': (np.exp(ret_erc_vt.cumsum()) /
                               np.exp(ret_erc_vt.cumsum()).cummax() - 1) * 100,
    'Equal Weight'         : (np.exp(ret_ew.cumsum()) /
                               np.exp(ret_ew.cumsum()).cummax() - 1) * 100,
    'Min Variance'         : (np.exp(ret_mv.cumsum()) /
                               np.exp(ret_mv.cumsum()).cummax() - 1) * 100,
}).reset_index()
dd_df.columns = ['Date','Risk Parity (ERC)',
                 'Risk Parity (Vol-Tgt)','Equal Weight','Min Variance']

ws3.merge_range('A1:E1', 'DRAWDOWN ANALYSIS (%)', fmt_title)
for col, h in enumerate(['Date','Risk Parity (ERC)',
                          'Risk Parity (Vol-Tgt)','Equal Weight','Min Variance']):
    ws3.write(1, col, h, fmt_header)

for row_idx, row in dd_df.iterrows():
    ws3.write_datetime(row_idx + 2, 0,
        pd.Timestamp(row['Date']).to_pydatetime(), fmt_date)
    for col_idx, col_name in enumerate(['Risk Parity (ERC)',
                                         'Risk Parity (Vol-Tgt)',
                                         'Equal Weight','Min Variance']):
        ws3.write(row_idx + 2, col_idx + 1,
                  round(float(row[col_name]), 4), fmt_number)

chart3 = wb.add_chart({'type': 'line'})
for i, (name, color) in enumerate(zip(names_xl, colors_xl)):
    chart3.add_series({
        'name'      : name,
        'categories': ['Drawdowns', 2, 0, len(dd_df)+1, 0],
        'values'    : ['Drawdowns', 2, i+1, len(dd_df)+1, i+1],
        'line'      : {'color': color, 'width': 1.5},
    })
chart3.set_title({'name': 'Drawdown (%)'})
chart3.set_x_axis({'name': 'Date', 'date_axis': True})
chart3.set_y_axis({'name': 'Drawdown (%)'})
chart3.set_legend({'position': 'bottom'})
chart3.set_size({'width': 720, 'height': 400})
ws3.insert_chart('G2', chart3)

print("   ✅ Sheet 3: Drawdowns + Chart")


# ════════════════════════════════════════════════════════════
# SHEET 4: ERC WEIGHTS + CHART
# ════════════════════════════════════════════════════════════
ws4 = wb.add_worksheet('ERC Weights')
ws4.set_tab_color('#2E7D32')
ws4.set_column('A:A', 14)
ws4.set_column('B:M', 14)

wt_erc_xl = (wt_erc * 100).round(2).reset_index()
asset_labels_xl = [LABELS.get(t, t) for t in ACTIVE_TICKERS]
wt_erc_xl.columns = ['Date'] + asset_labels_xl

ws4.merge_range(0, 0, 0, len(asset_labels_xl),
                'ERC PORTFOLIO WEIGHTS HISTORY (%)', fmt_title)
for col, h in enumerate(['Date'] + asset_labels_xl):
    ws4.write(1, col, h, fmt_header)

for row_idx, row in wt_erc_xl.iterrows():
    ws4.write_datetime(row_idx + 2, 0,
        pd.Timestamp(row['Date']).to_pydatetime(), fmt_date)
    for col_idx, label in enumerate(asset_labels_xl):
        ws4.write(row_idx + 2, col_idx + 1,
                  float(row[label]), fmt_pct)

# Stacked area chart for weights
chart4 = wb.add_chart({'type': 'area', 'subtype': 'stacked'})
area_colors = ['#4FC3F7','#81C784','#FFB74D','#F06292','#FFD54F',
               '#CE93D8','#80DEEA','#FF8A65','#B0BEC5','#FFCC02',
               '#A5D6A7','#EF9A9A']
n_wt_rows = len(wt_erc_xl)

for i, (label, color) in enumerate(zip(asset_labels_xl, area_colors)):
    chart4.add_series({
        'name'      : label,
        'categories': ['ERC Weights', 2, 0, n_wt_rows+1, 0],
        'values'    : ['ERC Weights', 2, i+1, n_wt_rows+1, i+1],
        'fill'      : {'color': color},
        'line'      : {'none': True},
    })
chart4.set_title({'name': 'ERC Weights Over Time (%)'})
chart4.set_x_axis({'name': 'Date', 'date_axis': True})
chart4.set_y_axis({'name': 'Weight (%)'})
chart4.set_legend({'position': 'bottom'})
chart4.set_size({'width': 820, 'height': 400})
ws4.insert_chart(2, len(asset_labels_xl) + 2, chart4)

print("   ✅ Sheet 4: ERC Weights + Stacked Chart")


# ════════════════════════════════════════════════════════════
# SHEET 5: RISK CONTRIBUTIONS + CHART
# ════════════════════════════════════════════════════════════
ws5 = wb.add_worksheet('Risk Contributions')
ws5.set_tab_color('#F57F17')
ws5.set_column('A:A', 14)
ws5.set_column('B:M', 14)

rc_xl = (rc_erc[ACTIVE_TICKERS] * 100).round(2).reset_index()
rc_xl.columns = ['Date'] + asset_labels_xl

ws5.merge_range(0, 0, 0, len(asset_labels_xl),
                'RISK CONTRIBUTIONS PER ASSET (%)', fmt_title)
for col, h in enumerate(['Date'] + asset_labels_xl):
    ws5.write(1, col, h, fmt_header)

for row_idx, row in rc_xl.iterrows():
    ws5.write_datetime(row_idx + 2, 0,
        pd.Timestamp(row['Date']).to_pydatetime(), fmt_date)
    for col_idx, label in enumerate(asset_labels_xl):
        ws5.write(row_idx + 2, col_idx + 1,
                  float(row[label]), fmt_pct)

chart5 = wb.add_chart({'type': 'bar', 'subtype': 'stacked'})
n_rc_rows = len(rc_xl)
for i, (label, color) in enumerate(zip(asset_labels_xl, area_colors)):
    chart5.add_series({
        'name'      : label,
        'categories': ['Risk Contributions', 2, 0, n_rc_rows+1, 0],
        'values'    : ['Risk Contributions', 2, i+1, n_rc_rows+1, i+1],
        'fill'      : {'color': color},
        'line'      : {'none': True},
    })
chart5.set_title({'name': 'Risk Contributions Over Time (%)'})
chart5.set_x_axis({'name': 'Date'})
chart5.set_y_axis({'name': 'Risk Contribution (%)'})
chart5.set_legend({'position': 'bottom'})
chart5.set_size({'width': 820, 'height': 400})
ws5.insert_chart(2, len(asset_labels_xl) + 2, chart5)

print("   ✅ Sheet 5: Risk Contributions + Chart")


# ════════════════════════════════════════════════════════════
# SHEET 6: STRESS TESTS
# ════════════════════════════════════════════════════════════
ws6 = wb.add_worksheet('Stress Tests')
ws6.set_tab_color('#880E4F')
ws6.set_column('A:A', 22)
ws6.set_column('B:F', 18)

ws6.merge_range('A1:F1',
    'STRESS TEST ANALYSIS — 2008 / 2020 / 2022', fmt_title)

row_cursor = 2
for period_name, df in stress_results.items():
    ws6.merge_range(row_cursor, 0, row_cursor, 4,
                    period_name, fmt_subheader)
    row_cursor += 1

    col_headers = ['Strategy'] + list(df.columns)
    for col, h in enumerate(col_headers):
        ws6.write(row_cursor, col, h, fmt_header)
    row_cursor += 1

    for strat_name, row in df.iterrows():
        ws6.write(row_cursor, 0, strat_name, fmt_label)
        for col_idx, val in enumerate(row.values):
            try:
                fmt = fmt_red if float(val) < 0 else fmt_green
                ws6.write(row_cursor, col_idx + 1, float(val), fmt)
            except:
                ws6.write(row_cursor, col_idx + 1, str(val), fmt_number)
        row_cursor += 1

    row_cursor += 1  # Gap between periods

print("   ✅ Sheet 6: Stress Tests")


# ════════════════════════════════════════════════════════════
# SHEET 7: CURRENT WEIGHTS SNAPSHOT + BAR CHART
# ════════════════════════════════════════════════════════════
ws7 = wb.add_worksheet('Current Snapshot')
ws7.set_tab_color('#4A148C')
ws7.set_column('A:A', 20)
ws7.set_column('B:F', 18)

latest_cov_xl = list(rolling_cov.values())[-1]
w_erc_now = optimize_erc(latest_cov_xl)
w_ew_now  = equal_weight(len(ACTIVE_TICKERS))
w_mv_now  = optimize_mean_variance(
    returns[ACTIVE_TICKERS].tail(252), latest_cov_xl
)
rc_now = compute_risk_contributions(w_erc_now, latest_cov_xl)

ws7.merge_range('A1:F1',
    f'CURRENT PORTFOLIO WEIGHTS SNAPSHOT', fmt_title)

snap_headers = ['Asset', 'Risk Parity %',
                'Equal Weight %', 'Min Variance %', 'Risk Contrib %']
for col, h in enumerate(snap_headers):
    ws7.write(1, col, h, fmt_header)

for row_idx, ticker in enumerate(ACTIVE_TICKERS):
    ws7.write(row_idx + 2, 0, LABELS.get(ticker, ticker), fmt_label)
    ws7.write(row_idx + 2, 1, round(w_erc_now[row_idx] * 100, 2), fmt_pct)
    ws7.write(row_idx + 2, 2, round(w_ew_now[row_idx]  * 100, 2), fmt_pct)
    ws7.write(row_idx + 2, 3, round(w_mv_now[row_idx]  * 100, 2), fmt_pct)
    ws7.write(row_idx + 2, 4, round(rc_now[row_idx]    * 100, 2), fmt_pct)

# Grouped bar chart comparing all 3 strategies
chart7 = wb.add_chart({'type': 'bar'})
n_assets_xl = len(ACTIVE_TICKERS)
chart7.add_series({
    'name'      : 'Risk Parity',
    'categories': ['Current Snapshot', 2, 0, n_assets_xl+1, 0],
    'values'    : ['Current Snapshot', 2, 1, n_assets_xl+1, 1],
    'fill'      : {'color': '#00E5FF'},
})
chart7.add_series({
    'name'      : 'Equal Weight',
    'categories': ['Current Snapshot', 2, 0, n_assets_xl+1, 0],
    'values'    : ['Current Snapshot', 2, 2, n_assets_xl+1, 2],
    'fill'      : {'color': '#FF6B6B'},
})
chart7.add_series({
    'name'      : 'Min Variance',
    'categories': ['Current Snapshot', 2, 0, n_assets_xl+1, 0],
    'values'    : ['Current Snapshot', 2, 3, n_assets_xl+1, 3],
    'fill'      : {'color': '#A5D6A7'},
})
chart7.set_title({'name': 'Current Weights by Strategy (%)'})
chart7.set_x_axis({'name': 'Weight (%)'})
chart7.set_y_axis({'name': 'Asset'})
chart7.set_legend({'position': 'bottom'})
chart7.set_size({'width': 600, 'height': 400})
ws7.insert_chart('G2', chart7)

print("   ✅ Sheet 7: Current Snapshot + Bar Chart")


# ════════════════════════════════════════════════════════════
# SHEET 8: ASSET STATISTICS + CHART
# ════════════════════════════════════════════════════════════
ws8 = wb.add_worksheet('Asset Statistics')
ws8.set_tab_color('#006064')
ws8.set_column('A:A', 20)
ws8.set_column('B:E', 16)

ws8.merge_range('A1:E1',
    'INDIVIDUAL ASSET STATISTICS (2006 → TODAY)', fmt_title)

stat_headers = ['Asset', 'Annual Return %',
                'Annual Vol %', 'Sharpe Ratio', 'Max Drawdown %']
for col, h in enumerate(stat_headers):
    ws8.write(1, col, h, fmt_header)

for row_idx, ticker in enumerate(ACTIVE_TICKERS):
    r  = returns[ticker]
    ann_ret = r.mean() * 252 * 100
    ann_vol = r.std() * np.sqrt(252) * 100
    sharpe  = r.mean() / r.std() * np.sqrt(252)
    cum     = np.exp(r.cumsum())
    max_dd  = (cum / cum.cummax() - 1).min() * 100

    ws8.write(row_idx + 2, 0, LABELS.get(ticker, ticker), fmt_label)
    ws8.write(row_idx + 2, 1, round(ann_ret, 2),
              fmt_green if ann_ret > 0 else fmt_red)
    ws8.write(row_idx + 2, 2, round(ann_vol, 2), fmt_number)
    ws8.write(row_idx + 2, 3, round(sharpe, 3),
              fmt_green if sharpe > 0.5 else fmt_number)
    ws8.write(row_idx + 2, 4, round(max_dd, 2), fmt_red)

# Bar chart: Sharpe ratios
chart8 = wb.add_chart({'type': 'bar'})
chart8.add_series({
    'name'      : 'Sharpe Ratio',
    'categories': ['Asset Statistics', 2, 0, n_assets_xl+1, 0],
    'values'    : ['Asset Statistics', 2, 3, n_assets_xl+1, 3],
    'fill'      : {'color': '#00E5FF'},
})
chart8.set_title({'name': 'Sharpe Ratio by Asset'})
chart8.set_x_axis({'name': 'Sharpe Ratio'})
chart8.set_legend({'none': True})
chart8.set_size({'width': 500, 'height': 380})
ws8.insert_chart('G2', chart8)

print("   ✅ Sheet 8: Asset Statistics + Chart")


# ════════════════════════════════════════════════════════════
# SHEET 9: DAILY RETURNS
# ════════════════════════════════════════════════════════════
ws9 = wb.add_worksheet('Daily Returns')
ws9.set_tab_color('#37474F')
ws9.set_column('A:A', 14)
ws9.set_column('B:E', 18)

ret_xl = pd.DataFrame({
    'Risk Parity (ERC)'    : ret_erc,
    'Risk Parity (Vol-Tgt)': ret_erc_vt,
    'Equal Weight'         : ret_ew,
    'Min Variance'         : ret_mv,
}).reset_index()
ret_xl.columns = ['Date','Risk Parity (ERC)',
                  'Risk Parity (Vol-Tgt)','Equal Weight','Min Variance']

ws9.merge_range('A1:E1', 'DAILY LOG RETURNS — ALL STRATEGIES', fmt_title)
for col, h in enumerate(ret_xl.columns):
    ws9.write(1, col, h, fmt_header)

for row_idx, row in ret_xl.iterrows():
    ws9.write_datetime(row_idx + 2, 0,
        pd.Timestamp(row['Date']).to_pydatetime(), fmt_date)
    for col_idx, col_name in enumerate(['Risk Parity (ERC)',
                                         'Risk Parity (Vol-Tgt)',
                                         'Equal Weight','Min Variance']):
        val = float(row[col_name])
        fmt = fmt_green if val > 0 else fmt_red
        ws9.write(row_idx + 2, col_idx + 1, round(val, 6), fmt)

print("   ✅ Sheet 9: Daily Returns (colour coded)")


# ════════════════════════════════════════════════════════════
# CLOSE AND SAVE
# ════════════════════════════════════════════════════════════
wb.close()

print(f"\n💾 Excel file saved:")
print(f"   {output_excel}")
print("""
┌──────────────────────────────────────────────────────┐
│         YOUR EXCEL FILE CONTAINS                     │
├──────────────────────────────────────────────────────┤
│  Tab 1: Summary Dashboard   — colour coded table     │
│  Tab 2: NAV Curves          — data + line chart      │
│  Tab 3: Drawdowns           — data + line chart      │
│  Tab 4: ERC Weights         — data + stacked chart   │
│  Tab 5: Risk Contributions  — data + stacked chart   │
│  Tab 6: Stress Tests        — 2008/2020/2022         │
│  Tab 7: Current Snapshot    — data + bar chart       │
│  Tab 8: Asset Statistics    — data + Sharpe chart    │
│  Tab 9: Daily Returns       — green/red colour coded │
└──────────────────────────────────────────────────────┘
""")
print("📥 TO DOWNLOAD:")
print("   1. Click the 📁 folder icon on the LEFT sidebar")
print("   2. Go to: drive → MyDrive → risk_parity_engine → outputs")
print("   3. Right-click 'Risk_Parity_Full.xlsx'")
print("   4. Click Download")
print("\n✅ Cell 17 complete — full Excel file ready!")


📊 Building complete Excel file with data + charts...
   ✅ Sheet 1: Summary Dashboard
   ✅ Sheet 2: NAV Curves + Chart
   ✅ Sheet 3: Drawdowns + Chart
   ✅ Sheet 4: ERC Weights + Stacked Chart
   ✅ Sheet 5: Risk Contributions + Chart
   ✅ Sheet 6: Stress Tests
   ✅ Sheet 7: Current Snapshot + Bar Chart
   ✅ Sheet 8: Asset Statistics + Chart
   ✅ Sheet 9: Daily Returns (colour coded)

💾 Excel file saved:
   /content/drive/MyDrive/risk_parity_engine/outputs/Risk_Parity_Full.xlsx

┌──────────────────────────────────────────────────────┐
│         YOUR EXCEL FILE CONTAINS                     │
├──────────────────────────────────────────────────────┤
│  Tab 1: Summary Dashboard   — colour coded table     │
│  Tab 2: NAV Curves          — data + line chart      │
│  Tab 3: Drawdowns           — data + line chart      │
│  Tab 4: ERC Weights         — data + stacked chart   │
│  Tab 5: Risk Contributions  — data + stacked chart   │
│  Tab 6: Stress Tests        — 2008/2020/2022         │
│  Ta